# ترجمة المحاضرة بـ V2: فصحى مبسّطة + قاموس مصطلحات تلقائي (Colab)

ما الجديد في V2 (القسم `[language]` في `configs/translation_v2.toml`):
1. **قاموس تلقائي:** نموذج لغوي يقرأ نص الفيديو كاملًا مرة واحدة قبل الترجمة ويقترح مقابلًا عربيًا واحدًا لكل مصطلح. يُحفظ في `cache/glossary/` ويمكن تعديله يدويًا.
2. **المولّدات تأخذ القاموس:** HY-MT1.5 بقالب «التدخل بالمصطلحات» الرسمي، وHy-MT2 وGemini بتعليمات المصطلحات وقواعد الفصحى المبسّطة.
3. **البوابة:** ترفض الترجمة التي لا تستخدم المصطلح المعتمد أو فيها كلمات عامية، وتسجّل الصياغات الحرفية.
4. **تحرير لغوي محروس (polish):** يزيل الصياغات الحرفية من الجملة المقبولة، ويُقبل فقط إذا نجح في البوابة كلها ولم يسُؤ MetricX بأكثر من 0.5.

- البيانات تبقى على Google Drive.
- التشغيل: Runtime → Change runtime type → **T4 GPU**، ثم الخلايا بالترتيب.
- قبل الخلية 2: أضف `GEMINI_API_KEY` في Secrets (أيقونة المفتاح على اليسار) وفعّل Notebook access.


In [ ]:
# 1) Drive + code (branch below) + requirements  (~2 min)
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

BRANCH = 'translation-v2-language'   # 'main' after the merge
LESSON = 'lesson_bfe775aa22957d44_academic_v2'
SUB = Path('outputs') / 'academic_fresh_pipeline' / LESSON
# Own folder, a shortcut in My Drive, the hidden shortcut-target folder Colab uses, or a shared drive.
_roots = [Path('/content/drive/MyDrive/ArabicDubbingV2')]
_roots += sorted(Path('/content/drive/.shortcut-targets-by-id').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/*/ArabicDubbingV2'))
RUN_DIR = next((r / SUB for r in _roots if (r / SUB).exists()), None)
if RUN_DIR is None:
    print('Drive mounted, but ArabicDubbingV2 is not visible. What this account sees:')
    print('  MyDrive:', sorted(p.name for p in Path('/content/drive/MyDrive').iterdir())[:40])
    _sc = Path('/content/drive/.shortcut-targets-by-id')
    print('  shortcut targets:', sorted(str(p.relative_to(_sc)) for p in _sc.glob('*/*'))[:20] if _sc.exists() else 'none')
    raise FileNotFoundError('Add a shortcut: drive.google.com -> Shared with me -> right-click ArabicDubbingV2 -> '
                            'Organize -> Add shortcut -> My Drive (in the SAME account Colab mounted), then rerun.')
_probe = RUN_DIR / 'translation_step' / '.write_test'
try:
    _probe.write_text('ok'); _probe.unlink()
except OSError:
    raise PermissionError('Drive folder is read-only for this account: share ArabicDubbingV2 as Editor.')
print('RUN_DIR =', RUN_DIR)

TR_REPO = Path('/content/translation_step_repo')
if TR_REPO.exists():
    subprocess.run(['git', '-C', str(TR_REPO), 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(TR_REPO), 'checkout', '-q', '-B', BRANCH, 'origin/' + BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git', str(TR_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu', 'sentencepiece', 'num2words'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'hunspell', 'hunspell-ar'], check=True, capture_output=True)
sys.path.insert(0, str(TR_REPO / 'translation'))
print(subprocess.run(['git', '-C', str(TR_REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# 2) secrets (run by hand: Colab asks for access the first time)
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('GEMINI_API_KEY loaded:', bool(os.environ['GEMINI_API_KEY']))


In [ ]:
# 3) V2 run (background thread; prints progress every minute; ~40-60 min on a T4)
import time
from dubbing_translation import colab as trc
from dubbing_translation.io import V2_CONFIG
JOB = trc.start(RUN_DIR, config=V2_CONFIG, use_hymt=True, hf_models=[('tencent/Hy-MT2-7B', 'hy_mt2_terms', 1)],
                use_llm=True, refine=True, rank=True, label='v2_language')
while not JOB['done']:
    print(time.strftime('%H:%M:%S'), '|', JOB['log'][-1] if JOB['log'] else 'starting...', flush=True)
    time.sleep(60)
trc.status(JOB, tail=5)
V2_DIR = JOB['result']['run_dir']
print('glossary of this video:', V2_DIR + '/glossary.json')


In [ ]:
# 4) V1 vs V2 on the same lecture
import pandas as pd
pd.DataFrame(trc.compare_v1_v2(RUN_DIR, V2_DIR)).T
